In [18]:
import os
import sqlite3
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

In [19]:
# Initialize the Dash app
app = dash.Dash(__name__)

# Define the layout of the app
app.layout = html.Div([
    html.H1("Collision Incidents by County and Year"),
    dcc.Dropdown(
        id='year-dropdown',
        options=[{'label': str(year), 'value': str(year)} for year in range(2020, 2025)],  # Example range of years
        value='2020',  # Default value
        clearable=False
    ),
    dcc.Graph(id='incident-bar-chart')
])

In [20]:
# Define a function to fetch data and create the Plotly figure
def get_figure(selected_year):
    # Connect to the SQLite database
    conn = sqlite3.connect(os.path.join(os.getcwd(), 'data/crash_data.db'))

    # Define the query
    query = f'''
    SELECT
        County,
        strftime("%Y", CollisionDate) AS CollisionYear,
        COUNT(*) AS IncidentCount
    FROM
        collision_incidents
    WHERE
        strftime("%Y", CollisionDate) = '{selected_year}'
    GROUP BY
        County, CollisionYear
    ORDER BY
        County, CollisionYear;
    '''

    # Execute the query and fetch the results into a DataFrame
    df = pd.read_sql_query(query, conn)

    # Close the database connection
    conn.close()

    # Create the bar chart using Plotly Express
    fig = px.bar(df, x='CollisionYear', y='IncidentCount', color='County', barmode='group',
                 labels={'CollisionYear': 'Year', 'IncidentCount': 'Number of Incidents'},
                 title=f'Number of Incidents by County for {selected_year}')

    return fig

In [21]:
# Define the callback to update the graph
@app.callback(
    Output('incident-bar-chart', 'figure'),
    [Input('year-dropdown', 'value')]
)
def update_graph(selected_year):
    return get_figure(selected_year)

In [22]:
# Run the app on a different port (e.g., 8051)
if __name__ == '__main__':
    app.run_server(debug=True, port=8051)